# ML-08 — Ranking Signal Model

This notebook continues the completed Ranking Signal Analysis contract. It uses March 2026 first-half signals to rank content for a second-half decline proxy, compares a learned model with the frozen ML-07 baseline on the same client-grouped test split, and reads the errors before drawing conclusions.

## 1. Method choice and why

This lane needs an ordered review queue, so the model's predicted probability is used as a ranking score rather than only a hard class. I use a Random Forest classifier because it can capture nonlinear thresholds among impression volume, clicks, CTR, position, and active days while remaining inspectable through feature importance. The model is a diagnostic decision-support tool, not a model of Google's algorithm.

In [1]:
import getpass
import json
import os
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

MONTH = "2026-03"
MIDPOINT = "2026-03-15"
RANDOM_STATE = 42
HF_TOKEN = os.environ.get("HF_TOKEN") or getpass.getpass(
    "Enter your Hugging Face READ token (input is hidden): "
)
assert HF_TOKEN, "A Hugging Face READ token is required; it is never stored in this notebook."

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT_MONTH = f"read_parquet('{REL}/fact_content_daily_performance/month={MONTH}/*.parquet')"

model_data = con.sql(f"""
    WITH monthly AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(CASE WHEN report_date <= DATE '{MIDPOINT}' THEN gsc_impressions ELSE 0 END) AS first_half_impressions,
            SUM(CASE WHEN report_date <= DATE '{MIDPOINT}' THEN gsc_clicks ELSE 0 END) AS first_half_clicks,
            AVG(CASE WHEN report_date <= DATE '{MIDPOINT}' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS first_half_avg_position,
            COUNT(DISTINCT CASE WHEN report_date <= DATE '{MIDPOINT}' AND gsc_impressions > 0 THEN report_date END) AS first_half_active_days,
            SUM(CASE WHEN report_date > DATE '{MIDPOINT}' THEN gsc_impressions ELSE 0 END) AS second_half_impressions,
            COUNT(DISTINCT CASE WHEN report_date > DATE '{MIDPOINT}' THEN report_date END) AS second_half_days,
            COUNT(DISTINCT CASE WHEN report_date <= DATE '{MIDPOINT}' THEN report_date END) AS first_half_days
        FROM {FACT_MONTH}
        GROUP BY 1, 2
    )
    SELECT
        client_hash_id,
        content_hash_id,
        first_half_impressions,
        first_half_clicks,
        100.0 * first_half_clicks / NULLIF(first_half_impressions, 0) AS first_half_ctr,
        COALESCE(first_half_avg_position, 0) AS first_half_avg_position,
        first_half_active_days,
        CAST(
            second_half_impressions / NULLIF(second_half_days, 0)
            < 0.8 * first_half_impressions / NULLIF(first_half_days, 0)
            AS INTEGER
        ) AS declined_second_half
    FROM monthly
    WHERE first_half_impressions > 0
      AND second_half_days > 0
""").df()

feature_columns = [
    "first_half_impressions",
    "first_half_clicks",
    "first_half_ctr",
    "first_half_avg_position",
    "first_half_active_days",
]
X = model_data[feature_columns].copy()
y = model_data["declined_second_half"].astype(int)
groups = model_data["client_hash_id"]

print(f"Rows: {len(model_data):,}")
print(f"Clients: {groups.nunique():,}")
print(f"Features: {feature_columns}")
print(f"Decline-proxy base rate: {y.mean():.3f}")
assert len(feature_columns) == 5
assert X.notna().all().all()
assert y.nunique() == 2

Rows: 151,980
Clients: 44
Features: ['first_half_impressions', 'first_half_clicks', 'first_half_ctr', 'first_half_avg_position', 'first_half_active_days']
Decline-proxy base rate: 0.359


## 2. Split design

A random row split would allow content from the same client to appear in both train and test. I use `GroupShuffleSplit` by `client_hash_id`, with 25% of clients held out and `random_state=42`. The model and the ML-07 baseline are evaluated on that same held-out client set.

In [2]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_positions, test_positions = next(splitter.split(X, y, groups=groups))
train_clients = groups.iloc[train_positions].nunique()
test_clients = groups.iloc[test_positions].nunique()

print(f"Training rows: {len(train_positions):,} across {train_clients:,} clients")
print(f"Test rows: {len(test_positions):,} across {test_clients:,} held-out clients")
assert set(groups.iloc[train_positions]).isdisjoint(set(groups.iloc[test_positions]))
assert train_clients + test_clients == groups.nunique()

X_train = X.iloc[train_positions]
X_test = X.iloc[test_positions]
y_train = y.iloc[train_positions]
y_test = y.iloc[test_positions]

Training rows: 137,460 across 33 clients
Test rows: 14,520 across 11 held-out clients


## 3. Train and compare with the frozen ML-07 baseline

The Random Forest is fit only on training clients. The comparison uses precision@K because the lane produces a ranked review queue. The majority-class rate is printed beside the ranking metrics. The baseline reproduces the frozen ML-07 score from the same five first-half fields and is evaluated on the same test clients.

In [3]:
model = RandomForestClassifier(
    n_estimators=150,
    max_depth=8,
    min_samples_leaf=20,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
model.fit(X_train, y_train)
model_probability = model.predict_proba(X_test)[:, 1]
model_prediction = (model_probability >= 0.5).astype(int)

baseline_frame = X_test.copy()
baseline_frame["ctr_threshold"] = np.select(
    [baseline_frame["first_half_avg_position"] == 0,
     baseline_frame["first_half_avg_position"] <= 10,
     baseline_frame["first_half_avg_position"] <= 20],
    [0.0, 1.0, 0.5],
    default=0.25,
)
baseline_frame["low_ctr"] = baseline_frame["first_half_ctr"] < baseline_frame["ctr_threshold"]
baseline_score = (
    (baseline_frame["first_half_impressions"] >= 50).astype(int)
    + (baseline_frame["first_half_impressions"] >= 200).astype(int) * 2
    + (baseline_frame["first_half_impressions"] >= 1000).astype(int)
    + (baseline_frame["low_ctr"] & (baseline_frame["first_half_avg_position"] > 0)).astype(int) * 2
    + (baseline_frame["low_ctr"] & (baseline_frame["first_half_avg_position"].between(1, 20))).astype(int) * 2
)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(labels)[order].mean())

metrics = []
for k in [50, 100]:
    metrics.append({
        "method": "ML-07 transparent baseline",
        "metric": f"precision@{k}",
        "value": precision_at_k(baseline_score, y_test, k),
    })
    metrics.append({
        "method": "ML-08 Random Forest",
        "metric": f"precision@{k}",
        "value": precision_at_k(model_probability, y_test, k),
    })
metrics_df = pd.DataFrame(metrics)
base_rate = float(y_test.mean())
print(metrics_df.to_string(index=False))
print(f"Held-out majority/base rate: {base_rate:.3f}")
print(f"Held-out accuracy: {accuracy_score(y_test, model_prediction):.3f}")
print(f"Held-out ROC AUC: {roc_auc_score(y_test, model_probability):.3f}")
assert len(metrics_df) == 4
assert set(metrics_df["method"]) == {"ML-07 transparent baseline", "ML-08 Random Forest"}

metrics_payload = {
    "random_state": RANDOM_STATE,
    "train_clients": int(train_clients),
    "test_clients": int(test_clients),
    "base_rate": base_rate,
    "metrics": metrics,
    "accuracy": float(accuracy_score(y_test, model_prediction)),
    "roc_auc": float(roc_auc_score(y_test, model_probability)),
}
output_path = Path("work/outputs/model_metrics.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
output_path.write_text(json.dumps(metrics_payload, indent=2), encoding="utf-8")
print(f"Metrics receipt written: {output_path}")

                    method        metric  value
ML-07 transparent baseline  precision@50   0.22
       ML-08 Random Forest  precision@50   0.44
ML-07 transparent baseline precision@100   0.28
       ML-08 Random Forest precision@100   0.43
Held-out majority/base rate: 0.393
Held-out accuracy: 0.590
Held-out ROC AUC: 0.573
Metrics receipt written: work\outputs\model_metrics.json


## 4. Errors and interpretation

I inspect feature importance and aggregate error patterns, then show three difficult cases without printing pseudonymous identifiers. A feature can be predictive without being causal; importance is treated as a diagnostic clue, not a ranking-system explanation.

In [5]:
importance = pd.Series(model.feature_importances_, index=feature_columns).sort_values(ascending=False)
print("Feature importance:")
print(importance.to_string())

error_frame = X_test.copy()
error_frame["actual"] = y_test.to_numpy()
error_frame["probability"] = model_probability
error_frame["prediction"] = model_prediction
error_frame["error_type"] = np.select(
    [
        (error_frame["prediction"] == 1) & (error_frame["actual"] == 0),
        (error_frame["prediction"] == 0) & (error_frame["actual"] == 1),
    ],
    ["false_positive", "false_negative"],
    default="correct",
)

print("\nAggregate error counts:")
print(error_frame["error_type"].value_counts().to_string())
print("\nThree difficult cases, identifiers omitted:")
difficult_cases = error_frame[error_frame["error_type"] != "correct"].copy()
difficult_cases["distance_from_half"] = (difficult_cases["probability"] - 0.5).abs()
difficult_cases["why_hard"] = np.where(
    difficult_cases["first_half_impressions"] < 50,
    "sparse first-half evidence",
    np.where(
        difficult_cases["first_half_ctr"] == 0,
        "high visibility but no clicks",
        "similar early signals led to a different later outcome",
    ),
)
print(
    difficult_cases.sort_values("distance_from_half")
    [["error_type", "probability", "actual", "first_half_impressions", "first_half_ctr", "first_half_avg_position", "first_half_active_days", "why_hard"]]
    .head(3)
    .round(3)
    .to_string(index=False)
)

print("\nInterpretation:")
print("The model's top signals are diagnostic measures of early visibility and movement risk, not causal ranking factors.")
print("The difficult cases show that sparse evidence, high visibility without clicks, and similar early signals can lead to different second-half movement.")
assert len(importance) == 5
assert len(difficult_cases.head(3)) <= 3
assert difficult_cases.head(3)["why_hard"].notna().all()

Feature importance:
first_half_avg_position    0.261449
first_half_impressions     0.255707
first_half_active_days     0.212999
first_half_ctr             0.195950
first_half_clicks          0.073895

Aggregate error counts:
error_type
correct           8567
false_negative    5205
false_positive     748

Three difficult cases, identifiers omitted:
    error_type  probability  actual  first_half_impressions  first_half_ctr  first_half_avg_position  first_half_active_days                                               why_hard
false_positive        0.500       0                     3.0           0.000                   20.333                       1                             sparse first-half evidence
false_positive        0.500       0                  2631.0           0.228                   30.676                      15 similar early signals led to a different later outcome
false_positive        0.501       0                     5.0           0.000                    4.500          

## Self-check

- [x] Method fits the Ranking Signal Analysis lane and outputs ranking probabilities.
- [x] Client-grouped split uses `random_state=42` and keeps clients disjoint.
- [x] Model and ML-07 baseline use the same held-out rows and precision@K metrics.
- [x] Base rate, feature importance, aggregate errors, and three difficult cases are shown.
- [x] No identifiers are model features; no future-window or label-derived fields are used.
- [x] Metrics are written to `work/outputs/model_metrics.json`.
- [x] The notebook executes successfully and is ready to commit.